In [1]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from data_models import MixtureClassificationModel,TeacherStudentModel, exponential_covariance, TwoModeTeacherStudentModel
from losses_regularizers import LogisticLoss, QuadraticRegularizer, PseudoHuberRegularizer,SquaredLoss,ElasticL1L2Loss
from erm_theory import ERMTrainer, TheoryFixedPointSolver

rng = np.random.default_rng(0)
rng2 = np.random.default_rng(1)

c = 5
n = c * 50
p = c * 10



mu = np.zeros(p); 
theta_star = np.zeros(p)
nb_ones = 10
for i in range(nb_ones):
    theta_star[i] = 1
delta = 1.95*np.ones((p,1))
d_v = delta.reshape((p,1))
pi = 0.4
C = 0.1*np.eye(p)+ pi*(1-pi)*d_v@d_v.T
models = dict()
model_names = ['Gaussian data', 'Non Gaussian data']
models['Non Gaussian data'] = TwoModeTeacherStudentModel(
    p=p,
    mu_x=mu,
    C_x=C,
    feature_dist="gaussian",
    theta_teacher=theta_star,
    pi = pi,
    delta = delta,
    noise_std = 0.2
)
models['Gaussian data'] = TeacherStudentModel(
    p=p,
    mu_x=mu,
    C_x=C,
    feature_dist="gaussian",
    theta_teacher=theta_star,
    noise_std = 0.2
)
lam = 5
reg = QuadraticRegularizer(a=np.zeros(p), H= lam *np.eye(p))


In [9]:
nb_points = 20
ratio_th_emp = 3
nb_points_th = ratio_th_emp*(nb_points-1)+1
etas = np.linspace(1,0, nb_points)
etas_th = np.linspace(1,0, nb_points_th)
K = models['Gaussian data'].class_params()['num_classes']
fp_damping =0.05
fp_tol = 2e-5
fp_max_iter = 1000
mc_samples = 100000
num_trials = 70
n_test = 3000

vars = {'Gaussian data': {}, 'Non Gaussian data': {}}
for key in vars.keys():
    vars[key] = {
        'error_emp_means': [], 'error_emp_stds': [],
        'error_th': [], 'error_gaussian_score': [], 
        'nu': [],'mu': [], 'kappa': [], 
        'alpha': [], 'A': []
    }
var_fixed_pt = ['mu', 'nu', 'kappa', 'alpha', 'A']
thetas_rcrd = dict()
for (idx,eta) in enumerate(etas_th):
    loss = ElasticL1L2Loss(eta)
    if idx%ratio_th_emp ==0:
        
        print(f"\n===== etabda = {eta:g} (empirical gen reg) =====")

        for model_name in model_names:
            model = models[model_name]
            trainer = ERMTrainer(model=model, loss=loss, regularizer=reg)
            emp = trainer.run_trials(
                n_train=n,
                n_test=n_test,
                num_trials=num_trials,
                rng=rng,
                solver_maxiter=2000,
                tol=1e-6,
                method="L-BFGS-B",
                verbose=False,
            )
            if eta==0 or eta ==1:
                thetas_rcrd[(model_name, int(eta))] = trainer.thetas
            vars[model_name]['error_emp_means'].append(emp["gen_loss_mean"])
            vars[model_name]['error_emp_stds'].append(emp["gen_loss_std"])
            print(f"Empirical loss for {model_name}: mean={emp['gen_loss_mean']:.6f}  std={emp['gen_loss_std']:.6f}")


    print(f"\n===== etabda = {eta:g} (theory) =====")
    for model_name in model_names:
        model = models[model_name]
        solver = TheoryFixedPointSolver(
            model=model,
            loss=loss,
            regularizer=reg,
            n_train=n,
            mc_samples=mc_samples,  # more samples for smaller etabda
            rng=np.random.default_rng(123),  # fixed for repeatability
        )

        th = solver.solve(
            max_iter=1000,
            tol= fp_tol,
            damping=np.max([lam,1]) * fp_damping,
            verbose=False,
            mu0=vars[model_name]['mu'][-1] if idx>0 else None,
            alpha0=vars[model_name]['alpha'][-1] if idx>0 else None,
            nu0=[1/(1+vars[model_name]['kappa'][-1][cl]) for cl in range(K)] if idx>0 else None,
        )

        vars[model_name]['error_th'].append(th["predicted_loss"])

        # Warm start for next etabda
        for var in var_fixed_pt:
            vars[model_name][var].append(th[var])
        print(f"Theory predicted loss for {model_name} model: {th['predicted_loss']:.6f}  (converged={th['converged']}, iters={th['num_iter']}, damping_final={th.get('damping_final', float('nan')):.3f})")
        
        # Prediction under score universality (can not use th['predicted_loss_gaussian_score'] directly in regression settings since y can take multiple values)
        z_samples = rng.standard_normal(size=100000)
        proj = (vars[model_name]['mu'][-1]-theta_star).reshape(-1,1)
        err_gauss_score_sple = proj.T@ model.mu_x+ np.sqrt(proj.T @ model.C_x @ proj + float(vars[model_name]['alpha'][-1][0])**2) * z_samples
        err_gauss_score = float(np.mean(loss.value(err_gauss_score_sple, 0)))
        vars[model_name]['error_gaussian_score'].append(err_gauss_score)
        print(f"Theory predicted for {model_name} under score universality: {err_gauss_score:.6f}")




===== etabda = 1 (empirical gen reg) =====


Empirical loss for Gaussian data: mean=2.233798  std=0.241700
Empirical loss for Non Gaussian data: mean=1.557971  std=0.081351

===== etabda = 1 (theory) =====
Theory predicted loss for Gaussian data model: 2.252216  (converged=True, iters=527, damping_final=0.011)
Theory predicted for Gaussian data under score universality: 2.241190
Theory predicted loss for Non Gaussian data model: 1.580086  (converged=True, iters=618, damping_final=0.010)
Theory predicted for Non Gaussian data under score universality: 1.444479

===== etabda = 0.982456 (theory) =====
Theory predicted loss for Gaussian data model: 2.118290  (converged=True, iters=489, damping_final=0.011)
Theory predicted for Gaussian data under score universality: 2.111562
Theory predicted loss for Non Gaussian data model: 1.547068  (converged=True, iters=607, damping_final=0.010)
Theory predicted for Non Gaussian data under score universality: 1.422072

===== etabda = 0.964912 (theory) =====
Theory predicted loss for Gaussian data

In [8]:
vars[model_name]['alpha'][-1]

array([0.16482451])

In [ ]:
cl=0
num_trials_th = 100000
scores_emp = dict()
scores_th = dict()
z_samples = rng2.standard_normal(size=num_trials_th) 
for idx,eta in zip([0,-1],[1,0]):
    for model_name in ['Gaussian data','Non Gaussian data']:
        sc_emp=[]
        for i in range(num_trials):
            Xte, yte = models[model_name].sample_class(cl, n_test, rng=rng)
            sc_emp += [Xte[j]@(thetas_rcrd[(model_name, eta)][i]-theta_star) for j in range(n_test)]
        scores_emp[(model_name, eta)] = np.array(sc_emp)
        Xtetst, ytetst = models[model_name].sample_class(cl,num_trials_th, rng=rng2)
        scores_th[(model_name, eta)] = Xtetst@(vars[model_name]["mu"][idx]-theta_star) + vars[model_name]["alpha"][idx][cl]*z_samples

In [ ]:

# --- Reset font settings for a modern, clean look ---
from matplotlib import patheffects
from sklearn.neighbors import KernelDensity
from scipy.stats import norm
from matplotlib.gridspec import GridSpec


# itm_max = len(vars[model_names[0]]['error_emp_means'])
# itm_max_th = len(vars[model_names[0]]['error_th'])-1

plt.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica']
plt.rcParams['font.size'] = 18
plt.rcParams['axes.linewidth'] = 1.5
plt.rcParams['xtick.major.width'] = 1.5
plt.rcParams['ytick.major.width'] = 1.5

# --- Define a refined, colorblind-friendly palette ---
# colors = {
#     'background': '#FFFFFF',
#     'grid': '#E0E0E0',
#     'empirical': '#D62728',    # Deep red
#     'Non Gaussian data': '#1F77B4',  # Deep blue
#     'Gaussian data': '#2CA02C',     # Green
#     'Non Gaussian data, score': '#9467BD', # Purple
#     'class_0': '#FF9999',      # Light red
#     'class_1': '#99CCFF',      # Light blue
#     'highlight': '#FF6347',    # Tomato Red
#     'accent_line': "#97B33C",  # Medium Sea Green
# }
colors = {
    'Gaussian data':{
        'raw_score': '#2CA02C',
        'score_univ': "#97B33C",
    },
    'Non Gaussian data':
    {
        'raw_score': '#1F77B4',
        'score_univ': '#9467BD',
    },
}


# --- Create the figure ---
# fig, axes = plt.subplots(1,2, figsize=(8, 18), facecolor=colors['background'])
# plt.subplots_adjust(wspace=0.25)
fig = plt.figure(figsize=(36, 16))

gs = GridSpec(
    nrows=2,
    ncols=3,
    height_ratios=[1, 1],  # adjust if needed
    width_ratios=[1, 2, 1],  # adjust if needed
    hspace=0.4,
    wspace=0.3
)

axes_histo = {0: {'Gaussian data' : fig.add_subplot(gs[0, 0]), 'Non Gaussian data': fig.add_subplot(gs[1, 0])},
              1: {'Gaussian data' : fig.add_subplot(gs[0, 2]), 'Non Gaussian data': fig.add_subplot(gs[1, 2])}}

ax_mid_upper  = fig.add_subplot(gs[0, 1])
ax_mid_bottom = fig.add_subplot(gs[1, 1])

# Bottom: spans both columns
# --- LEFT PANEL: Histograms + PDFs ---
# for c_label in classes:
c_label=0
alpha = {'Non Gaussian data':0.6,'Gaussian data':0.2}
for eta in [0,1]:
    for model_name in model_names:
        axes_histo[eta][model_name].hist(
            scores_emp[(model_name, eta)],
            bins=30,
            density=True,
            alpha=alpha[model_name],
            color=colors[model_name]['raw_score'],
            edgecolor='white',
            linewidth=2,
            label=f'{model_name} score' 
        )
        scth = scores_th[(model_name, eta)]
        x = np.linspace(np.mean(scth) - 4 * np.std(scth), np.mean(scth) + 4 * np.std(scth), 500)
        kde = KernelDensity(kernel='gaussian', bandwidth=0.2).fit(scth.reshape(-1, 1))
        log_dens = kde.score_samples(x.reshape(-1, 1))
        axes_histo[eta][model_name].plot(
            x,
            np.exp(log_dens),
            color=colors[model_name]['raw_score'],
            linewidth=3.5,
            linestyle='-.',
            label=f'Theoretical density',
            path_effects=[patheffects.withStroke(linewidth=6, foreground='white', alpha=0.8)]
        )
        axes_histo[eta][model_name].plot(
            x,
            norm.pdf(x, np.mean(scth),  np.std(scth)),
            linestyle='--',
            color=colors[model_name]['score_univ'],
            linewidth=2.5,
            label=f'Gaussian fit',
            alpha=0.8
        )

    axes_histo[eta]['Gaussian data'].set_title(r"Score histograms $x^\top(\mu-\theta^*)$ for $\eta =$ {eta}".format(eta=eta), pad=20, fontweight='bold')
    axes_histo[eta]['Non Gaussian data'].set_xlabel("Decision Score", labelpad=10)
    for model_name in model_names:
        axes_histo[eta][model_name].set_ylabel("Density", labelpad=10)
        axes_histo[eta][model_name].legend(loc='lower right', frameon=True, framealpha=1, edgecolor='white')
    # axes_histo[eta].grid(True, linestyle='-', color=colors['grid'], alpha=0.3)


# --- MIDDLE UPPER PANEL: Classification Error ---
for model_name in model_names:

    ax_mid_upper.plot(
        etas_th,
        vars[model_name]['error_th'],
        color=colors[model_name]['raw_score'],
        linewidth=3.5,
        label=f'Theoretical Error {model_name}',
        path_effects=[patheffects.withStroke(linewidth=5, foreground='white', alpha=0.5)]
    )
    ax_mid_upper.plot(
        etas,
        vars[model_name]['error_emp_means'],
        color=colors[model_name]['raw_score'],
        linestyle ='none',
        linewidth=3.5,
        label=f'Empirical Error {model_name}',
        marker='o',
        markersize=9,
        markerfacecolor='white',
        markeredgewidth=2,
        markeredgecolor=colors[model_name]['raw_score']
        # colors['empirical']
    )
    ax_mid_upper.plot(
        etas_th,
        vars[model_name]['error_gaussian_score'],
        color=colors[model_name]['score_univ'],
        linestyle ='--',
        linewidth=3.5,
        label=f'Pred. score gaussian hyp',
        path_effects=[patheffects.withStroke(linewidth=5, foreground='white', alpha=0.5)]
    )

# --- Styling ---
ax_mid_upper.set_title(r"Classification Error vs elastic parameter Loss $= (1- \eta) \ell_2 + \eta \ell_1$", pad=20, fontweight='bold')
# ax_mid_upper.set_xlabel("Regularization Parameter ($\lambda$)", labelpad=10)
ax_mid_upper.set_ylabel("Classification Error", labelpad=10)
ax_mid_upper.legend(loc='upper left', frameon=True, framealpha=0.9, edgecolor='white', title='Error Curves')
# ax_mid_upper.grid(True, linestyle='-', color=colors['grid'], alpha=0.3)

# --- MIDDLE BOTTOM PANEL: fixed point quantities ---
display_vars =['alpha', 'nu', '1/(1+kappa)']
linestyle ={'alpha':'-', 'nu':'--', '1/(1+kappa)': '-.'}
labels ={'alpha':r'$\alpha_\ast$', 'nu':r'$\nu_\ast$', '1/(1+kappa)': r'$\frac{1}{1+\kappa_\ast}$'}
for model_name in model_names:
    vars[model_name]['1/(1+kappa)'] = [1/(1+kap) for kap in vars[model_name]['kappa']]
    for var in display_vars:
        ax_mid_bottom.plot(
            etas_th,
            vars[model_name][var],
            color=colors[model_name]['raw_score'],
            linewidth=3.5,
            linestyle = linestyle[var],
            label=f'{labels[var]} for {model_name}',
            path_effects=[patheffects.withStroke(linewidth=5, foreground='white', alpha=0.5)]
        )



# --- Styling ---
ax_mid_bottom.set_title("", pad=20, fontweight='bold')
ax_mid_bottom.set_xlabel(r"$\longleftarrow \ell_2 \  Loss  \qquad \qquad \qquad \qquad \qquad \qquad\eta \qquad \qquad \qquad \qquad \qquad \qquad \qquad \ell_1 \  Loss \ \longrightarrow $", labelpad=10)
ax_mid_bottom.set_ylabel("Asymptotic description variables", labelpad=10)
ax_mid_bottom.legend(loc='upper right', frameon=True, framealpha=0.9, edgecolor='white', title='Error Curves')
# ax_mid_bottom.grid(True, linestyle='-', color=colors['grid'], alpha=0.3)


# --- Save in high quality ---
plt.tight_layout()
# fig.savefig("ICML_teaser_figure.pdf", bbox_inches='tight', dpi=1200, transparent=True)
# fig.savefig("ICML_teaser_figure.png", bbox_inches='tight', dpi=1200, transparent=True)
plt.show()
